### Coleta de dados de Temperatura, Umidade e Precipitação

Será utilizado o dataset derived-era5-single-levels-daily-statistics do ERA5

Documentação em:
https://cds.climate.copernicus.eu/datasets/derived-era5-single-levels-daily-statistics?tab=documentation

Os dados extraídos serão:
<pre>
- Precipitação  -> variável total_precipitation, está variável é retornada em metros, será necessário conveter para milimetros (dividir por 1000)
</pre>

Os dados serão coletados por Ano e Mês 

Os dados requisitados estão no retangulo geográfico geográfico [6, -74, -34, -35] -> [Norte, Oeste, Sul, Leste] em graus onde está o Brasil




In [1]:
import cdsapi
import os
import xarray as xr

In [2]:
import os, sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
# Cria a conexão Spark

# 1. Remove qualquer barreira de proxy local que jogue o tráfego para a rede da empresa
os.environ.pop('HTTP_PROXY', None)
os.environ.pop('HTTPS_PROXY', None)
os.environ.pop('http_proxy', None)
os.environ.pop('https_proxy', None)

# 2. Garante que o Spark use o Python correto do venv
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 3. Força o IP local estrito
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# 4. Inicializa configurando a autenticação local do Worker
spark = SparkSession.builder \
    .appName("TesteLocal") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.network.auth.enabled", "false") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [4]:
# Os arquivos utilizados durante o processamento serão removidos no final do notebook
remover_arquivos = []

Requisição dos dados da variável derived-era5-single-levels-daily-statistics do ERA5 utilizando a biblioteca cdsapi

In [5]:
dataset = "derived-era5-single-levels-daily-statistics"
request = {
    "product_type": "reanalysis",
    "variable": [
        "total_precipitation"
    ],
    "year": "2025",
    "month": ["01"],
    "day": ["01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12",
            "13", "14", "15"
    ],
    "daily_statistic": "daily_mean",
    "time_zone": "utc-03:00",
    "frequency": "1_hourly",
    "area": [6, -74, -34, -35] # Retangulo definido por [Norte, Oeste, Sul, Leste] em graus onde está o Brasil
}

# Informações de autenticação estão em:
# C:\Users\DRT90628\.ecmwfdatastoresrc
# *** Criar um novo contrato de autenticação deverá ser criado usando um usuário de serviços do Einstein

client = cdsapi.Client(
    url = os.getenv("ECMWF_DATASTORES_URL"),
    key = os.getenv("ECMWF_DATASTORES_KEY"),
)

ret_download = client.retrieve(dataset, request).download()
remover_arquivos.append(ret_download)

print(f"Download completed: {ret_download}")

2026-07-21 16:58:47,156 WARNING [2026-06-08T00:00:00] An issue with the following parameters has been identified:

- `maximum_2m_temperature_since_previous_post_processing`
- `minimum_2m_temperature_since_previous_post_processing`
- `10m_wind_gust_since_previous_post_processing`
- `maximum_total_precipitation_rate_since_previous_post_processing`
- `minimum_total_precipitation_rate_since_previous_post_processing`

Please refer to the documented [Known issue](https://confluence.ecmwf.int/display/CKB/ERA5+family+post-processed+daily+statistics+documentation#ERA5familypostprocesseddailystatisticsdocumentation-KnownIssues) for details. **Data downloaded for the parameters listed above should not be used.** A fix is to be released as soon as possible. Please [follow updates on this topic on our Forum announcement](https://forum.ecmwf.int/t/issue-affecting-some-parameters-from-the-era5-post-processed-daily-statistics-on-single-levels/15057).
2026-07-21 16:58:47,156 INFO Request ID is 44596e7e

Download completed: 55f0332f0eba2f5df1405b1b8a505c3.nc


In [ ]:
# import zipfile

# zip_path = r"C:\Marco Conti\Projetos\MAIS-v2\Ondas_Calor\{file_name}".format(file_name = ret_download)
# extract_path = r"C:\Marco Conti\Projetos\MAIS-v2\Ondas_Calor"

# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_path)


# print(f"Files extracted to {extract_path}")

Dicionário utilizado durante a conversão dos arquvos .nc para Spark Dataframe

In [ ]:
# files_data_variables = [
#     {
#         "source_file": "2m_dewpoint_temperature_stream-oper_daily-mean.nc",
#         "data_variable": "d2m",
#         "indicador": "ponto_orvalho",
#         "df_name": "df_ponto_orvalho",
#         "target_file": "ponto_orvalho.csv",
#         "unidade_original": "kelvin",
#         "unidade_destino": "celsius"
#     },
#     {
#         "source_file": "2m_temperature_0_daily-mean.nc",
#         "data_variable": "t2m",
#         "indicador": "temperatura",
#         "df_name": "df_temperatura",
#         "target_file": "temperatura_t2m.csv",
#         "unidade_original": "kelvin",
#         "unidade_destino": "celsius"
#     },
#     {
#         "source_file": "total_precipitation_0_daily-mean.nc",
#         "data_variable": "tp",
#         "indicador": "precipitacao",
#         "df_name": "df_precipitacao",
#         "target_file": "precipitacao.csv",
#         "unidade_original": "metros",
#         "unidade_destino": "milimetros"
#     }
# ]

# def get_file_attribute(source_file, attribute):
#     for item in files_data_variables:
#         if item["source_file"] == source_file:
#             return item.get(attribute)
#     return None

Esta função ira converter os dados dos arqivos .nc para o format Dask para então converter para Dataframe Spark <br>
Isso deixa o processamento em paralelo e será importante para processamento de grandes volumes (1 ano com todos os meses e dias)

In [6]:
def convert_netcdf4_Spark(file_name):
    with xr.open_dataset(f"C:\\Marco Conti\\Projetos\\MAIS-v2\\Ondas_Calor\\{file_name}"
                        ,engine="netcdf4"
                        ,chunks={"time": 365
                                ,"latitude": 100
                                ,"longitude": 100 }
                        ) as ds:
        
        # Transforma o Dataset em um Spark Dataframe
        df_dask   = ds.to_dask_dataframe()
        df_dask_c = df_dask.compute()
        df_spark  = spark.createDataFrame(df_dask_c)    
    return df_spark

In [7]:
df_precipitacao = convert_netcdf4_Spark(ret_download)

C:\Users\DRT90628\AppData\Local\Temp\ipykernel_10596\854711052.py:2: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 100. This could degrade performance. Instead, consider rechunking after loading.
  with xr.open_dataset(f"C:\\Marco Conti\\Projetos\\MAIS-v2\\Ondas_Calor\\{file_name}"
C:\Users\DRT90628\AppData\Local\Temp\ipykernel_10596\854711052.py:2: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 100. This could degrade performance. Instead, consider rechunking after loading.
  with xr.open_dataset(f"C:\\Marco Conti\\Projetos\\MAIS-v2\\Ondas_Calor\\{file_name}"
c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Marco Conti\Projetos\MAIS-v

In [10]:
drop_cols = ["valid_time", "number", "tp"]
df_precipitacao = \
    (df_precipitacao
        .withColumns({"indicador": F.lit("precipitacao")
                     ,"valor": (F.col("tp") * F.lit(1000)).cast('double')
                     ,"unidade_medida": F.lit("celsius")
                     ,"data_medicao": F.col("valid_time").cast("date")}
                    )
        .drop(*drop_cols)
    )

df_precipitacao.printSchema()
df_precipitacao.show()

root
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- indicador: string (nullable = false)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = false)
 |-- data_medicao: date (nullable = true)

+--------+---------+------------+--------------------+--------------+------------+
|latitude|longitude|   indicador|               valor|unidade_medida|data_medicao|
+--------+---------+------------+--------------------+--------------+------------+
|     6.0|    -74.0|precipitacao|  0.2705852093640715|       celsius|  2025-01-01|
|     6.0|   -73.75|precipitacao|  0.2659956517163664|       celsius|  2025-01-01|
|     6.0|    -73.5|precipitacao| 0.11265277862548828|       celsius|  2025-01-01|
|     6.0|   -73.25|precipitacao|  0.1548926084069535|       celsius|  2025-01-01|
|     6.0|    -73.0|precipitacao| 0.13987223792355508|       celsius|  2025-01-01|
|     6.0|   -72.75|precipitacao|0.007828076377336401|       celsius|  2025-01-01

In [11]:

df_precipitacao = \
       (df_precipitacao
            .select("data_medicao"
                   ,"latitude"
                   ,"longitude"
                   ,"indicador"
                   ,"valor"
                   ,"unidade_medida"))

In [13]:
# df_precipitacao.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_precipitacao.csv", index=False)

df_precipitacao.toPandas().to_parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_precipitacao.parquet")

c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [14]:
# df_csv = spark.read.csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_umidade_temperatura_precipitacao.csv", header=True, inferSchema=True)
# print("df_csv:", df_csv.count())
# df_csv.printSchema()
# df_csv.show(10,False)


df_parquet = spark.read.parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_precipitacao.parquet")
print("df_csv:", df_parquet.count())
df_parquet.printSchema()
df_parquet.show(10,False)


df_csv: 379155
root
 |-- data_medicao: date (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)

+------------+--------+---------+------------+--------------------+--------------+
|data_medicao|latitude|longitude|indicador   |valor               |unidade_medida|
+------------+--------+---------+------------+--------------------+--------------+
|2025-01-01  |6.0     |-74.0    |precipitacao|0.2705852093640715  |celsius       |
|2025-01-01  |6.0     |-73.75   |precipitacao|0.2659956517163664  |celsius       |
|2025-01-01  |6.0     |-73.5    |precipitacao|0.11265277862548828 |celsius       |
|2025-01-01  |6.0     |-73.25   |precipitacao|0.1548926084069535  |celsius       |
|2025-01-01  |6.0     |-73.0    |precipitacao|0.13987223792355508 |celsius       |
|2025-01-01  |6.0     |-72.75   |precipitacao|0.007828076377336401|c

In [15]:
# **** INCLUIR EXCLUSÃO DE ARQUIVOS (.zip e .nc)

for file in remover_arquivos:
    print("Arquivo:", file, end="")
    os.remove(file)
    print(" Removido com sucesso")


Arquivo: 55f0332f0eba2f5df1405b1b8a505c3.nc Removido com sucesso
